<a href="https://colab.research.google.com/github/Spandana2704/DL/blob/main/WEEK12_DL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Character-Level RNN**

In [9]:
import torch, torch.nn as nn

text = "hello world"
chars = list(set(text))
c2i = {c:i for i,c in enumerate(chars)}
i2c = {i:c for c,i in c2i.items()}

data = [c2i[c] for c in text]
X = torch.tensor([data[:-1]])
Y = torch.tensor(data[1:])

embed = nn.Embedding(len(chars), 8)
rnn = nn.RNN(8, 16, batch_first=True)
fc = nn.Linear(16, len(chars))

opt = torch.optim.Adam(list(embed.parameters())+list(rnn.parameters())+list(fc.parameters()), lr=0.01)

for _ in range(200):
    out,_ = rnn(embed(X))
    loss = nn.CrossEntropyLoss()(fc(out).view(-1,len(chars)), Y.view(-1))
    opt.zero_grad(); loss.backward(); opt.step()

# ✅ Test
test = "hell"
t = torch.tensor([[c2i[c] for c in test]])
out,_ = rnn(embed(t))
pred = fc(out)[0,-1]
print("Next char:", i2c[pred.argmax().item()])

Next char: o


**Word-Level RNN**

In [10]:
text = "hello world hello ai"
words = text.split()

vocab = list(set(words))
w2i = {w:i for i,w in enumerate(vocab)}
i2w = {i:w for w,i in w2i.items()}

data = [w2i[w] for w in words]
X = torch.tensor([data[:-1]])
Y = torch.tensor(data[1:])

embed = nn.Embedding(len(vocab), 8)
rnn = nn.RNN(8, 16, batch_first=True)
fc = nn.Linear(16, len(vocab))

opt = torch.optim.Adam(list(embed.parameters())+list(rnn.parameters())+list(fc.parameters()), lr=0.01)

for _ in range(200):
    out,_ = rnn(embed(X))
    loss = nn.CrossEntropyLoss()(fc(out).view(-1,len(vocab)), Y.view(-1))
    opt.zero_grad(); loss.backward(); opt.step()

# ✅ Test
test = ["hello","world"]
t = torch.tensor([[w2i[w] for w in test]])
out,_ = rnn(embed(t))
pred = fc(out)[0,-1]
print("Next word:", i2w[pred.argmax().item()])

Next word: hello


**LSTM**

In [11]:
embed = nn.Embedding(len(vocab), 8)
lstm = nn.LSTM(8, 16, batch_first=True)
fc = nn.Linear(16, len(vocab))

opt = torch.optim.Adam(list(embed.parameters())+list(lstm.parameters())+list(fc.parameters()), lr=0.01)

for _ in range(200):
    out,_ = lstm(embed(X))
    loss = nn.CrossEntropyLoss()(fc(out).view(-1,len(vocab)), Y.view(-1))
    opt.zero_grad(); loss.backward(); opt.step()

# ✅ Test
out,_ = lstm(embed(t))
print("LSTM next word:", i2w[fc(out)[0,-1].argmax().item()])

LSTM next word: hello


**GRU**

In [12]:
embed = nn.Embedding(len(vocab), 8)
gru = nn.GRU(8, 16, batch_first=True)
fc = nn.Linear(16, len(vocab))

opt = torch.optim.Adam(list(embed.parameters())+list(gru.parameters())+list(fc.parameters()), lr=0.01)

for _ in range(200):
    out,_ = gru(embed(X))
    loss = nn.CrossEntropyLoss()(fc(out).view(-1,len(vocab)), Y.view(-1))
    opt.zero_grad(); loss.backward(); opt.step()

# ✅ Test
out,_ = gru(embed(t))
print("GRU next word:", i2w[fc(out)[0,-1].argmax().item()])

GRU next word: hello


**Encoder–Decoder**

In [17]:
import torch, torch.nn as nn

pairs = [("hi","hello")]

chars = list(set("<>" + "".join([x+y for x,y in pairs])))
c2i = {c:i for i,c in enumerate(chars)}
i2c = {i:c for c,i in c2i.items()}

def t(s): return torch.tensor([[c2i[c] for c in s]])

E = nn.Embedding(len(chars), 8)
L = nn.LSTM(8, 16, batch_first=True)
F = nn.Linear(16, len(chars))

opt = torch.optim.Adam(list(E.parameters())+list(L.parameters())+list(F.parameters()), lr=0.01)
loss_fn = nn.CrossEntropyLoss()

# 🔹 TRAIN
for _ in range(500):
    for x,y in pairs:
        _,(h,c) = L(E(t(x)))
        inp = t("<")
        loss = 0
        for ch in y:
            out,(h,c) = L(E(inp),(h,c))
            loss += loss_fn(F(out).view(1,-1), t(ch).view(-1))
            inp = t(ch)
        opt.zero_grad(); loss.backward(); opt.step()

# 🔹 TEST
_,(h,c) = L(E(t("hi")))
inp = t("<")
res = ""

for _ in range(6):
    out,(h,c) = L(E(inp),(h,c))
    p = F(out).argmax(2)
    ch = i2c[p.item()]
    res += ch
    inp = p

print("Output:", res)

Output: helloo


**Attention**

In [19]:
import torch
import torch.nn as nn

class Attention(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, hidden, encoder_outputs):
        hidden = hidden.permute(1,0,2)   # (batch,1,hidden)

        scores = torch.bmm(hidden, encoder_outputs.transpose(1,2))  # (batch,1,seq_len)

        weights = torch.softmax(scores, dim=2)   # ✅ FIX HERE

        context = torch.bmm(weights, encoder_outputs)  # (batch,1,hidden)

        return context, weights

att = Attention()

encoder_outputs = torch.rand(1,5,8)
hidden = torch.rand(1,1,8)

context, weights = att(hidden, encoder_outputs)

print("weights:", weights)
print("sum:", weights.sum())   # should be 1

weights: tensor([[[0.1419, 0.2957, 0.2157, 0.1924, 0.1543]]])
sum: tensor(1.0000)
